<a href="https://colab.research.google.com/github/abhsrivastava/hugging_face_transformers/blob/main/AutoModels.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Load dependencies
%pip install -q --upgrade transformers



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 16.2 MB/s eta 0:00:00


In [3]:
# Imports

from transformers import (
    AutoModel,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    AutoConfig
)
import torch

print('✅ Imports Complete')
print(f'PyTorch Version: {torch.__version__}')

✅ Imports Complete
PyTorch Version: 2.11.0+cpu


In [4]:
# Load the Task specific class

CHECKPOINT = "distilbert-base-uncased-finetuned-sst-2-english"

# Load Tokenizer (this remains same irrespective of the model used)

tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)

print(f'Tokenizer Loaded {tokenizer.__class__.__name__}')

model = AutoModelForSequenceClassification.from_pretrained(CHECKPOINT)

print(f'Model Loaded: {model.__class__.__name__}')

# Set Model to Eval Mode

model.eval()

print(f'Model Architecture: {model.config.architectures}')


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Tokenizer Loaded BertTokenizer


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Model Loaded: DistilBertForSequenceClassification
Model Architecture: ['DistilBertForSequenceClassification']


In [6]:
# Now let us inspect the config

config = model.config

print(f'Model Configuration')
print('=' * 50)
print(f'Model Type: {config.model_type}')
print(f'Number of Labels: {config.num_labels}')
print(f'Hidden Size: {config.hidden_size}')
print(f'Number of Layers: {config.num_hidden_layers}')
print(f'Attention Heads: {config.num_attention_heads}')
print(f'Vocabulary Size: {config.vocab_size:,}')
print(f'Max Position Embeddings: {config.max_position_embeddings}')


Model Configuration
Model Type: distilbert
Number of Labels: 2
Hidden Size: 768
Number of Layers: 6
Attention Heads: 12
Vocabulary Size: 30,522
Max Position Embeddings: 512


In [8]:
# Print Label Mappings

print('Label Mappings')
print("=" * 50)
print(f'Id2Label: {config.id2label}')
print(f'Label2Id: {config.label2id}')
print("Interpretation")
for idx, label in config.id2label.items():
  print(f"Output Neuron {idx} maps to label '{label}'")

Label Mappings
Id2Label: {0: 'NEGATIVE', 1: 'POSITIVE'}
Label2Id: {'NEGATIVE': 0, 'POSITIVE': 1}
Interpretation
Output Neuron 0 maps to label 'NEGATIVE'
Output Neuron 1 maps to label 'POSITIVE'


In [16]:
# Text we want the sentiment-classification model to evaluate
text = "NovaPay's mobile app keeps crashing whenever I try to add a new Card!"

# Convert the text into the PyTorch tensors expected by the model.
# model_inputs is a dictionary containing:
#   - input_ids: numerical token IDs
#   - attention_mask: identifies real tokens versus padding
model_inputs = tokenizer(text, return_tensors="pt")

print(f"Input Text: {text!r}")
print(f"Token Count: {model_inputs['input_ids'].shape[1]}")

# Run the model in inference mode.
#
# model.eval() was called earlier to disable training-specific behavior.
# torch.no_grad() prevents PyTorch from recording calculations for training.
# We only want a prediction, so gradients are unnecessary.
with torch.no_grad():
    model_outputs = model(**model_inputs)

# Logits are the model's raw scores for each possible label.
#
# For this model:
#   position 0 = NEGATIVE
#   position 1 = POSITIVE
#
# Logits are not probabilities. They can be negative and do not sum to 1.
logits = model_outputs.logits

print(f"\nLogits: {logits}")
print(f"Shape: {logits.shape} (batch_size=1, num_labels=2)")

# Convert the raw logits into probabilities using softmax.
#
# dim=-1 means softmax is applied across the final dimension—the two labels.
# The resulting probabilities are between 0 and 1 and sum to 1.
probabilities = torch.softmax(logits, dim=-1)

# Find the position of the label with the highest probability.
# .item() converts the one-value PyTorch tensor into a Python integer.
predicted_label_id = torch.argmax(probabilities, dim=-1).item()

# Translate the numeric label ID into its human-readable label.
predicted_label = config.id2label[predicted_label_id]

# Get the probability assigned to the selected label.
confidence = probabilities[0, predicted_label_id].item()

print(f"Prediction: {predicted_label}")
print(f"Confidence: {confidence:.4f}")
print(f"Negative: {probabilities[0, 0].item():.4f}")
print(f"Positive: {probabilities[0, 1].item():.4f}")

Input Text: "NovaPay's mobile app keeps crashing whenever I try to add a new Card!"
Token Count: 20

Logits: tensor([[ 3.3867, -2.7258]])
Shape: torch.Size([1, 2]) (batch_size=1, num_labels=2)
Prediction: NEGATIVE
Confidence: 0.9978
Negative: 0.9978
Positive: 0.0022


In [19]:
# Silent Falure Model (What happends when we load the wrong class)

model_wrong = AutoModel.from_pretrained(CHECKPOINT)
model_wrong.eval()
print(f"Model Loaded: {model_wrong.__class__.__name__}")
print(f"Model Architectures: {model_wrong.config.architectures}")

params_correct = sum(p.numel() for p in model.parameters())
params_wrong = sum(p.numel() for p in model_wrong.parameters())
head_size = params_correct - params_wrong

print(f'AutoModelForSequenceClassification: {params_correct:>12,} parameters')
print(f'AutoModel (base): {params_wrong:>12} parameters')
print(f'Difference (The classification head ): {head_size:>12,} parameters')
print(f'The classification head has {head_size:>12,} parameters without which you cannot classify text')


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased-finetuned-sst-2-english
Key                   | Status     |  | 
----------------------+------------+--+-
classifier.weight     | UNEXPECTED |  | 
pre_classifier.weight | UNEXPECTED |  | 
classifier.bias       | UNEXPECTED |  | 
pre_classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model Loaded: DistilBertModel
Model Architectures: ['DistilBertForSequenceClassification']
AutoModelForSequenceClassification:   66,955,010 parameters
AutoModel (base):     66362880 parameters
Difference (The classification head ):      592,130 parameters
The classification head has      592,130 parameters without which you cannot classify text
